# Experiment Tracking with MLflow
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajit-ai/Data_Science/blob/main/11_MLOps_Deployment/mlflow_experiment_tracking.ipynb)

"Which of my 30 runs used max_depth=7?" - MLflow logs parameters, metrics, artifacts and models for every experiment, queryable from a UI.

Fully local/free here (`file:` store); same code points to a tracking server later.

In [ ]:
!pip install -q mlflow scikit-learn

## 1. Manual logging of a parameter sweep

In [ ]:
import mlflow
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target,
                                      stratify=data.target, random_state=42)

mlflow.set_experiment("breast-cancer-rf")

for n_est, depth in [(100, None), (300, None), (300, 5), (500, 10)]:
    with mlflow.start_run():
        mlflow.log_params({"n_estimators": n_est, "max_depth": depth})
        rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth,
                                    random_state=42).fit(Xtr, ytr)
        auc = roc_auc_score(yte, rf.predict_proba(Xte)[:, 1])
        mlflow.log_metric("test_auc", auc)
        mlflow.sklearn.log_model(rf, "model")
        print(f"n={n_est} depth={depth} auc={auc:.4f}")

## 2. Autologging - one line covers everything

In [ ]:
mlflow.sklearn.autolog()
with mlflow.start_run(run_name="autologged"):
    rf2 = RandomForestClassifier(random_state=42).fit(Xtr, ytr)
mlflow.sklearn.autolog(disable=True)

## 3. Query runs programmatically

In [ ]:
runs = mlflow.search_runs(experiment_ids=[mlflow.get_experiment_by_name(
              "breast-cancer-rf").experiment_id],
          order_by=["metrics.test_auc DESC"])
cols = ["tags.mlflow.runName", "params.n_estimators", "params.max_depth", "metrics.test_auc"]
print(runs[cols].head(5).to_string(index=False))

## 4. Launch the UI

In [ ]:
ui_note = """
# local:
mlflow ui --backend-store-uri ./mlruns   ->  http://localhost:5000

# Colab:
!pip install -q pyngrok
from pyngrok import ngrok
ngrok.set_auth_token("YOUR_NGROK_TOKEN")
ngrok.connect(5000)
get_ipython().system_raw("mlflow ui --port 5000 &")
"""
print(ui_note)

**Why bother**
- Compare runs side-by-side instead of scrolling notebook outputs.
- Reproduce: params + git commit + artifact = full recipe.
- `log_model` gives instant serving hooks (`mlflow models serve`).
- Team scale: point `mlflow.set_tracking_uri("http://server:5000")` at shared backend - code unchanged.